# MAE-ViT: Analysis & Visualization
Explore pre-training reconstructions, attention maps, and fine-tuning results.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from models import MaskedAutoencoder, ViTClassifier
from utils import build_dataset, build_dataloader
from utils.visualization import visualize_reconstruction, denormalize

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using: {device}')

## 1. Load Pre-trained MAE

In [ ]:
import yaml
with open('../configs/pretrain.yaml') as f:
    cfg = yaml.safe_load(f)

model = MaskedAutoencoder(
    image_size=cfg['data']['image_size'],
    patch_size=cfg['model']['patch_size'],
    embed_dim=cfg['model']['embed_dim'],
    encoder_depth=cfg['model']['encoder_depth'],
    encoder_heads=cfg['model']['encoder_heads'],
    decoder_embed_dim=cfg['model']['decoder_embed_dim'],
    decoder_depth=cfg['model']['decoder_depth'],
    decoder_heads=cfg['model']['decoder_heads'],
).to(device)

ckpt_path = '../experiments/mae_pretrain/checkpoints/best.pth'
if Path(ckpt_path).exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')
else:
    print('No checkpoint found — run pre-training first.')

## 2. Reconstruction Visualization

In [ ]:
import math
from einops import rearrange

val_ds = build_dataset('val', mode='pretrain', image_size=32)
val_dl = build_dataloader(val_ds, batch_size=16, num_workers=0, shuffle=True)
imgs, labels = next(iter(val_dl))

model.eval()
with torch.no_grad():
    loss, pred, mask = model(imgs.to(device))
    print(f'Reconstruction loss: {loss.item():.4f}')
    
    grid_size = int(math.sqrt(mask.shape[1]))
    patch_size = cfg['model']['patch_size']
    pred_imgs = rearrange(
        pred, 'b (h w) (p1 p2 c) -> b c (h p1) (w p2)',
        h=grid_size, w=grid_size, p1=patch_size, p2=patch_size, c=3
    )

n = 6
fig, axes = plt.subplots(n, 3, figsize=(9, 3*n))
for i in range(n):
    orig = denormalize(imgs[i]).permute(1, 2, 0).numpy()
    rec  = denormalize(pred_imgs[i].cpu()).permute(1, 2, 0).numpy()
    mask_img = mask[i].reshape(grid_size, grid_size)
    mask_img = mask_img.unsqueeze(0).repeat(3, 1, 1)
    mask_img = torch.nn.functional.interpolate(mask_img.unsqueeze(0).float(), scale_factor=patch_size).squeeze(0)
    masked = (denormalize(imgs[i]) * (1 - mask_img.cpu())).permute(1, 2, 0).numpy()
    
    for ax, img, title in zip(axes[i], [orig, masked, rec], ['Original', 'Masked', 'Reconstructed']):
        ax.imshow(img.clip(0, 1))
        ax.axis('off')
        if i == 0: ax.set_title(title, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Attention Map Visualization (Last Encoder Block)

In [ ]:
# Hook into the last attention layer to extract attention weights
attention_weights = {}

def hook_fn(module, input, output):
    # Re-compute attention weights from Q, K
    x = input[0]
    B, N, C = x.shape
    qkv = module.qkv(x).reshape(B, N, 3, module.num_heads, module.head_dim)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, _ = qkv.unbind(0)
    attn = (q @ k.transpose(-2, -1)) * module.scale
    attention_weights['last'] = attn.softmax(dim=-1).detach().cpu()

handle = model.encoder.blocks[-1].attn.register_forward_hook(hook_fn)

img = imgs[:1].to(device)
model.eval()
with torch.no_grad():
    model(img)
handle.remove()

attn = attention_weights['last'][0]   # (heads, N_vis, N_vis)
print(f'Attention shape: {attn.shape}')

n_heads = attn.shape[0]
fig, axes = plt.subplots(1, n_heads, figsize=(3 * n_heads, 3))
for h, ax in enumerate(axes):
    ax.imshow(attn[h].numpy(), cmap='viridis')
    ax.set_title(f'Head {h}')
    ax.axis('off')
plt.suptitle('Self-Attention Maps (Last Encoder Block, Visible Patches)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Fine-tuning Results

In [ ]:
import csv

def load_log(path):
    rows = []
    with open(path) as f:
        for row in csv.DictReader(f):
            rows.append({k: float(v) for k, v in row.items()})
    return rows

pretrain_log = load_log('../experiments/mae_pretrain/log.csv')
finetune_log = load_log('../experiments/mae_finetune/log.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pre-train loss
epochs_p = [r['epoch'] for r in pretrain_log]
axes[0].plot(epochs_p, [r['train_loss'] for r in pretrain_log], label='Train')
axes[0].plot(epochs_p, [r['val_loss'] for r in pretrain_log], label='Val')
axes[0].set_title('MAE Reconstruction Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Fine-tune accuracy
epochs_f = [r['epoch'] for r in finetune_log]
axes[1].plot(epochs_f, [r['val_acc1'] for r in finetune_log], label='Top-1')
axes[1].plot(epochs_f, [r['val_acc5'] for r in finetune_log], label='Top-5')
axes[1].set_title('Fine-tune Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Fine-tune loss
axes[2].plot(epochs_f, [r['train_loss'] for r in finetune_log], color='orange')
axes[2].set_title('Fine-tune Train Loss'); axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Best Top-1 Accuracy: {max(r['val_acc1'] for r in finetune_log):.2f}%")

## 5. t-SNE Feature Space Visualization

In [ ]:
from sklearn.manifold import TSNE
import yaml

with open('../configs/finetune.yaml') as f:
    ft_cfg = yaml.safe_load(f)

classifier = ViTClassifier(
    image_size=32, patch_size=4, embed_dim=384, depth=6, num_heads=6,
    num_classes=100, global_pool=True
).to(device)

ft_ckpt = '../experiments/mae_finetune/checkpoints/best.pth'
if Path(ft_ckpt).exists():
    ckpt = torch.load(ft_ckpt, map_location=device)
    classifier.load_state_dict(ckpt['model_state'])

# Extract features for 1000 samples (10 classes x 100)
val_ds_ft = build_dataset('val', mode='finetune', image_size=32)
val_dl_small = build_dataloader(val_ds_ft, batch_size=128, num_workers=0, shuffle=True)

all_feats, all_labels = [], []
classifier.eval()
with torch.no_grad():
    for imgs, labels in val_dl_small:
        feats = classifier.encoder(imgs.to(device)).mean(dim=1)
        all_feats.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())
        if sum(len(f) for f in all_feats) >= 1000:
            break

feats = np.concatenate(all_feats)[:1000]
labels_np = np.concatenate(all_labels)[:1000]

print('Running t-SNE...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
emb = tsne.fit_transform(feats)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(emb[:, 0], emb[:, 1], c=labels_np, cmap='tab20', s=8, alpha=0.7)
plt.colorbar(scatter, label='Class')
plt.title('t-SNE of MAE Encoder Features (CIFAR-100 Validation)')
plt.axis('off')
plt.tight_layout()
plt.show()